In [ ]:
# Celda 1
# Control against the 2024 mesh of the Ayuntamiento (built from its oblique aerial photos): what the mesh has and the
# model does not ("falta": a wall, a gable, a body), and what the model has and the mesh does not ("sobra": an invented
# row of windows, a wall too high). Both surfaces are sampled every SPACING m and each sample takes its distance to the
# other surface. Runs after 14_materiales and 18_malla_referencia (it reuses the tiles and their placement).
%cd ..

VIEWS_JSON = "sources/malla_madrid_2024/vistas.json"   # Written by 18: tiles, matrix, floor
MODEL_GLB = "blender/colegio_imperial_lod3.glb"
GPKG = "data/catastro_manzana.gpkg"
ORIGIN = (440000.0, 4473700.0, 640.0)
CONJUNTO = ["0139402VK4703G", "0139424VK4703G", "0139423VK4703G"]   # Parcels of the ensemble
SPACING = 0.35        # m between samples of the mesh
MODEL_SPACING = 0.5   # m between samples of the model (it has many hidden faces; memory)
FOOTPRINT_BUFFER = 1.0   # Samples within this distance of the footprints of the ensemble (eaves, cornices)
ABOVE_GROUND = 1.5    # Only samples this high over the street (cars, trees, people at the bottom)
GAP = 0.8             # A sample farther than this from the other surface is a difference (m)
VOXEL = 1.0           # Differences are grouped by 1 m voxels touching each other
MIN_AREA = 4.0        # Groups smaller than this (m2 of surface) are noise
SAMPLES_NPZ = "data/control/malla_distancias.npz"
REPORT = "data/control/malla_diferencias.csv"
MAP = "renders/control_malla.png"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"

In [ ]:
# Celda 2
import os
import csv
import json
import subprocess
import numpy as np
import shapely
import geopandas as gpd
from scipy import ndimage
from PIL import Image, ImageDraw

parts = gpd.read_file(GPKG, layer="building_parts")
footprint = shapely.union_all([shapely.affinity.translate(g, -ORIGIN[0], -ORIGIN[1])
                               for g, ref in zip(parts.geometry, parts["localId"]) if str(ref).split("_")[0] in CONJUNTO])
area = footprint.buffer(FOOTPRINT_BUFFER + 2).bounds
views = json.load(open(VIEWS_JSON, encoding="utf-8"))
config = {"glbs": views["glbs"], "matrix": views["matrix"], "floor_box": views["floor_box"], "floor_z": views["floor_z"],
          "model": os.path.abspath(MODEL_GLB), "area": [area[0], area[2], area[1], area[3]], "spacing": SPACING, "model_spacing": MODEL_SPACING,
          "out": os.path.abspath(SAMPLES_NPZ)}
os.makedirs(os.path.dirname(SAMPLES_NPZ), exist_ok=True)
with open("data/control/malla_distancias.json", "w", encoding="utf-8") as f:
    json.dump(config, f)
run = subprocess.run([BLENDER, "-b", "--factory-startup", "--python", "scripts/malla_distancias.py", "--",
                      "data/control/malla_distancias.json"], capture_output=True, text=True)
for line in run.stdout.splitlines():
    if line.startswith(("Tiles", "Vertical", "Triangles", "Saved")) or "Error" in line:
        print(line)
if run.returncode != 0 or "Saved" not in run.stdout:
    print(run.stdout[-2000:], run.stderr[-2000:])
    raise RuntimeError("Blender failed")

In [ ]:
# Celda 3
# Keep the samples on the ensemble (inside its footprints plus the buffer, above the street) and group the
# differences. Ground height under each sample: the lowest mesh sample within 3 m (the street or the courtyard).
data = np.load(SAMPLES_NPZ)
near = footprint.buffer(FOOTPRINT_BUFFER)

def on_ensemble(s):
    inside = shapely.contains_xy(near, s[:, 0], s[:, 1])
    return s[inside]

mesh_s, model_s = on_ensemble(data["mesh"]), on_ensemble(data["model"])
ground_cells = {}
for x, y, z in data["mesh"][:, :3]:
    key = (int(x // 3), int(y // 3))
    ground_cells[key] = min(ground_cells.get(key, 1e9), z)
def ground(s):
    return np.array([min(ground_cells.get((int(x // 3) + i, int(y // 3) + j), 1e9) for i in (-1, 0, 1) for j in (-1, 0, 1))
                     for x, y in s[:, :2]])
mesh_s = mesh_s[mesh_s[:, 2] > ground(mesh_s) + ABOVE_GROUND]
model_s = model_s[model_s[:, 2] > ground(model_s) + ABOVE_GROUND]
# Column 7: distance along the outward normal of the mesh. A mesh sample outside the model (> 0) is something the
# model lacks; a model sample outside the mesh (> 0) is something the model adds. Model faces hidden inside the
# volume (walls between bays, prisms under the towers) are inside the mesh (< 0) and do not count.
missing = mesh_s[(mesh_s[:, 6] > GAP) & (mesh_s[:, 7] > 0)]
extra = model_s[(model_s[:, 6] > GAP) & (model_s[:, 7] > 0)]
print(f"mesh: {len(mesh_s)} samples, {np.mean(mesh_s[:, 6] <= GAP):.0%} within {GAP} m of the model, median {np.median(mesh_s[:, 6]):.2f} m")
print(f"falta: {len(missing) * SPACING ** 2:.0f} m2 of mesh outside the model | sobra: {len(extra) * SPACING ** 2:.0f} m2 of model outside the mesh")

def groups(bad, kind):
    if len(bad) == 0:
        return []
    idx = np.floor(bad[:, :3] / VOXEL).astype(int)
    lo = idx.min(0)
    grid = np.zeros(idx.max(0) - lo + 1, dtype=bool)
    grid[tuple((idx - lo).T)] = True
    labels, count = ndimage.label(grid, structure=np.ones((3, 3, 3)))
    sample_label = labels[tuple((idx - lo).T)]
    out = []
    for k in range(1, count + 1):
        g = bad[sample_label == k]
        surface = len(g) * SPACING ** 2
        if surface < MIN_AREA:
            continue
        n = g[:, 3:6].mean(0)
        orientation = "cubierta" if abs(n[2]) > 0.6 else "muro"
        if orientation == "muro":
            facing = ["E", "N", "O", "S"][int(((np.degrees(np.arctan2(n[1], n[0])) + 45) % 360) // 90)]
            orientation += " hacia " + facing
        out.append({"tipo": kind, "superficie_m2": round(surface, 1), "orientacion": orientation,
                    "x": round(float(g[:, 0].mean()), 1), "y": round(float(g[:, 1].mean()), 1),
                    "z_min": round(float(g[:, 2].min()) + ORIGIN[2], 1), "z_max": round(float(g[:, 2].max()) + ORIGIN[2], 1),
                    "distancia_mediana": round(float(np.median(g[:, 6])), 2),
                    "xmin": round(float(g[:, 0].min()), 1), "xmax": round(float(g[:, 0].max()), 1),
                    "ymin": round(float(g[:, 1].min()), 1), "ymax": round(float(g[:, 1].max()), 1)})
    return sorted(out, key=lambda r: -r["superficie_m2"])

report = groups(missing, "falta en el modelo") + groups(extra, "sobra en el modelo")
for k, r in enumerate(report):
    r["n"] = k + 1
with open(REPORT, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(report[0].keys()))
    w.writeheader()
    w.writerows(report)
print(len(report), "groups of differences ->", REPORT)
for r in report[:30]:
    print(f"  {r['n']:2d} {r['tipo']:20s} {r['superficie_m2']:7.1f} m2  {r['orientacion']:15s} x {r['x']:6.1f} y {r['y']:6.1f}  z {r['z_min']}-{r['z_max']}  d {r['distancia_mediana']}")

In [ ]:
# Celda 4
# Map from above: footprint in grey; samples of the mesh missing in the model in red, samples of the model missing
# in the mesh in blue (the higher, the darker).
SCALE = 8   # px per m
x0, y0, x1, y1 = footprint.buffer(3).bounds
W, H = int((x1 - x0) * SCALE), int((y1 - y0) * SCALE)
img = Image.new("RGB", (W, H), "white")
draw = ImageDraw.Draw(img)
def px(x, y):
    return ((x - x0) * SCALE, (y1 - y) * SCALE)
for poly in getattr(footprint, "geoms", [footprint]):
    draw.polygon([px(*p) for p in poly.exterior.coords], fill=(225, 225, 225), outline=(150, 150, 150))
    for hole in poly.interiors:
        draw.polygon([px(*p) for p in hole.coords], fill="white", outline=(150, 150, 150))
zmax = max(mesh_s[:, 2].max(), model_s[:, 2].max())
for bad, color in ((extra, (40, 80, 220)), (missing, (220, 40, 40))):
    for x, y, z in bad[np.argsort(bad[:, 2]), :3]:
        t = 0.35 + 0.65 * z / zmax
        c = tuple(int(255 - (255 - v) * t) for v in color)
        u, v = px(x, y)
        draw.rectangle([u - 1, v - 1, u + 1, v + 1], fill=c)
for r in report[:30]:
    u, v = px(r["x"], r["y"])
    draw.text((u + 4, v - 4), str(r["n"]), fill="black")
img.save(MAP)
print("Map:", MAP, "| numbers = rows of the report")

In [ ]:
# Celda 5
# One view per large group, mesh and model with the same camera: from outside along the normal of the group, 25 m
# away and 12 m above (from above for roofs). Side by side in renders/control_malla/.
from PIL import Image
OUT = "renders/control_malla/"
os.makedirs(OUT, exist_ok=True)
DIRECTIONS = {"E": (1, 0), "N": (0, 1), "O": (-1, 0), "S": (0, -1)}
views_cfg = {}
for r in report[:16]:
    center = (r["x"], r["y"], (r["z_min"] + r["z_max"]) / 2 - ORIGIN[2])
    if r["orientacion"] == "cubierta":
        camera = (center[0] + 6, center[1] - 14, center[2] + 22)
    else:
        dx, dy = DIRECTIONS[r["orientacion"][-1]]
        camera = (center[0] + dx * 25 + dy * 6, center[1] + dy * 25 - dx * 6, center[2] + 12)
    views_cfg[f"{r['n']:02d}"] = [camera, center]
cfg = {**{k: views[k] for k in ("glbs", "matrix", "floor_box", "floor_z")}, "model": os.path.abspath(MODEL_GLB),
       "area": config["area"], "views": views_cfg, "lens": 35, "size": [900, 700], "out": os.path.abspath(OUT) + os.sep}
with open("data/control/malla_vistas.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f)
run = subprocess.run([BLENDER, "-b", "--factory-startup", "--python", "scripts/vistas_malla.py", "--",
                      "data/control/malla_vistas.json"], capture_output=True, text=True)
if run.returncode != 0 or "Saved" not in run.stdout:
    print(run.stdout[-2000:], run.stderr[-2000:])
    raise RuntimeError("Blender failed")
for name in views_cfg:
    a, b = Image.open(OUT + name + ".jpg"), Image.open(OUT + name + "_modelo.jpg")
    both = Image.new("RGB", (a.width + b.width, a.height), "white")
    both.paste(a, (0, 0))
    both.paste(b, (a.width, 0))
    both.save(OUT + name + "_lado_a_lado.jpg", quality=85)
    os.remove(OUT + name + ".jpg")
    os.remove(OUT + name + "_modelo.jpg")
print(len(views_cfg), "views in", OUT)